# LERM Parameter Grid: Precision/Recall/F1 vs. m, target_density, w_size, w_incre, block_size, alpha (red/snr3)

Single-method, single-condition diagnostic: LERM (Laplacian Eigenmaps of Recurrence
Matrices, James et al. 2024 -- `ammonyte.TimeEmbeddedSeries` / `RecurrenceMatrix` /
`RQARes.lerm_transitions`), run on the **red/snr3** noise condition only
(`noise_sensitivity_data/noise_red_snr3.npz`, 20 seeds). Mirrors the same methodology as
`KS/ks_grid_red_snr3.ipynb` / `Pelt/pelt_grid_red_snr3.ipynb` / `Dynp/dynp_grid_red_snr3.ipynb`
-- same data, same ground truth, same train/test split, same `evaluate_detection` metric --
adapted to LERM's own, very different parameter set.

**LERM is a staged pipeline, not one function call.** KS/Pelt/Dynp each grid the parameters of a
single function. LERM is: time-delay embedding -> `find_epsilon` density search -> recurrence
matrix -> Laplacian eigenmaps -> block-smoothing -> bootstrap-threshold crossing detection. That
staging is *why* the parameter list below looks so different from the other notebooks' grids --
most of LERM's named arguments are not free "sensitivity" parameters at all, they're either
auto-estimated from the data or settings of an internal search procedure. Deciding which is
which (and why) took a full back-and-forth read of `ammonyte`'s actual source before writing this
grid; the reasoning is recorded here rather than in a separate document because it's load-bearing
for every choice below.

**Parameter disposition (the full design, settled parameter by parameter):**

| Parameter | Disposition | Why |
|---|---|---|
| `tau` | auto (never set) | `TimeEmbeddedSeries(series, m, tau=None)` estimates it per-series via `tau_search` (first minimum of mutual information). No reason to override an estimator the package already ships. |
| `epsilon` | computed, never set | `find_epsilon` solves for it given `target_density`. It is not portable across `m` -- confirmed on this data: median embedded pairwise distance goes 0.69 -> 1.17 -> 1.51 for m=3,7,11 on the same seed (distance in embedded space grows roughly as sqrt(m)), so a single fixed epsilon would leave the recurrence matrix nearly empty at one end of the `m` grid and nearly saturated at the other. `target_density` is the portable, comparable quantity; epsilon is what `find_epsilon` derives from it and the data. |
| `eps` (start) | fixed *recipe*, scaled with `m` | Only a starting guess -- `find_epsilon`'s search converges to the same `target_density` regardless of where it starts. Scaled by `sqrt(m/M_REFERENCE)` (see `lerm_scoring.eps_start_for_m`) purely so convergence doesn't take needlessly more iterations at the grid's extreme `m` values -- doesn't change what's found, only how fast. |
| `tolerance` (density search) | fixed | `find_epsilon` stops at the *first* epsilon whose density is within this band of `target_density` -- it does not keep searching for the closest match. Too loose and neighboring `TARGET_DENSITY_GRID` points blur into each other; too tight and the search takes more iterations at the pipeline's most expensive stage. Set below to roughly a third of `TARGET_DENSITY_GRID`'s smallest gap. |
| `amp` | fixed | `find_epsilon`'s search step size (`range_finder`'s `miss*amp`). Purely a solver-convergence-speed knob; only worth revisiting if Section 8's benchmark shows the search struggling to converge. |
| `n_samples` (bootstrap draws) | fixed | Monte Carlo resolution for the percentile estimate inside `confidence_interval` -- doesn't change what's being estimated, just how precisely. Package default (10,000) leaves ~50 points below the tightest tail this grid tests (alpha=0.01 -> 0.5th percentile); raise it if `ALPHA_GRID` is later pushed tighter. |
| `w` (bootstrap draw size) | fixed *rule*, value derived per grid point | `confidence_interval` draws `w` values from the **smoothed FI series**, whose length is itself downstream of `w_size`/`w_incre`/`block_size` -- three grid axes. A flat `w` would be a sensible fraction of a long FI series at one grid point and most of the entire series at another. Set as `max(W_MIN, len(smoothed_series) * W_FRACTION)`, computed fresh per grid point in `lerm_scoring.score_one`. |
| `m`, `target_density`, `w_size`, `w_incre`, `block_size` | **gridded** | No auto-estimator, no default that's obviously right for this dataset, and each plausibly changes detection quality. The train grid below. |
| `upper`, `lower` | collapsed into one gridded axis, `alpha` | Both are just percentiles read off the *same* bootstrap distribution of means (`ammonyte.utils.sampling.confidence_interval`) -- gridding them independently would allow nonsensical asymmetric combinations for no benefit. Reparameterized symmetrically as a single significance level: `lower = 100*(alpha/2)`, `upper = 100*(1-alpha/2)`. This is LERM's exact analog of KS's `d_c` -- a pure post-hoc threshold on an already-computed series, so (like `d_c`) it's cheap to grid jointly *and* swept densely on the test pool as the precision/recall tradeoff curve. |

**Staged caching, not a flat `joblib.Memory` per call.** Unlike Dynp (whose cache is a flat
memoization of one expensive function), LERM's cost is concentrated in the embed -> `find_epsilon`
-> eigendecomposition stage, which only depends on `(m, target_density, w_size, w_incre, seed)` --
`block_size` and `alpha` are cheap post-processing on top of it. `lerm_scoring.py`'s
`compute_fi_series` is the disk-cached boundary: it runs once per `(m, target_density, w_size,
w_incre, seed)` and is reused for every `block_size` x `alpha` combination scored against it (16x
reuse at the grid sizes below). See that module's docstring for the one known limitation (the
cache sits after `laplacian_eigenmaps`, not before it, so `w_size`/`w_incre` still trigger a fresh
eigendecomposition rather than reusing one across the `w_size`/`w_incre` grid).

**Grid sizing is a first pass, not a committed budget.** LERM's expensive stage -- an O(n^2)
density search feeding an O(n^3) eigendecomposition on a ~6000-point embedded series -- has no
cheap analog in KS/Pelt/Dynp to calibrate against, so unlike those notebooks' grids (sized against
a benchmarked single-call cost before being written down), the ranges below are a starting point
that Section 8's benchmark must validate *before* committing to the full run. Expect to shrink
`M_GRID`/`TARGET_DENSITY_GRID`/`W_SIZE_GRID`/`W_INCRE_GRID` (the four axes that hit the expensive
stage) if the benchmark says otherwise -- `BLOCK_SIZE_GRID` and `ALPHA_GRID` are cheap regardless,
since they never trigger a new `compute_fi_series` call.


In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ammonyte as amt
from ammonyte.utils.metrics import evaluate_detection
from joblib import Parallel, delayed

## Section 1 -- Configuration

In [ ]:
DATA_PATH   = "../../noise_sensitivity_data/noise_red_snr3.npz"
DETECTION_TOLERANCE = 0.5   # +/- 0.5 kyr matching tolerance, matches KS/Pelt/Dynp
RANDOM_SEED = 42
N_JOBS = -1   # CARC run: use all allocated cores; lower this if running locally

# Standard seed budget -- same as KS uses (LERM has no rbf-like heavy-cost outlier
# needing a smaller budget of its own).
BUDGET = dict(pool_size=20, n_train=14, n_test=6)

# ---- Fixed pipeline settings: not gridded (see intro markdown for why each is here
# instead of in GRIDS below). ----
EPS_START_BASE = 1.0        # find_epsilon starting guess at m=M_REFERENCE
M_REFERENCE    = 5          # eps_start scaled by sqrt(m/M_REFERENCE) for other m (lerm_scoring.eps_start_for_m)
DENSITY_TOLERANCE = 0.01    # find_epsilon convergence band -- ~1/3 of TARGET_DENSITY_GRID's smallest gap (0.03)
SEARCH_AMP = 10             # find_epsilon search step amplitude (package default)
N_SAMPLES_BOOTSTRAP = 10000 # bootstrap resamples for confidence_interval (package default)
W_FRACTION = 0.125          # bootstrap draw size = this fraction of the smoothed FI series length ...
W_MIN = 10                  # ... floored at this many points (lerm_scoring._bootstrap_w)

# ---- Grid: expensive stage (embed -> find_epsilon -> recurrence matrix -> eigenmaps).
# FIRST PASS ONLY -- validate against Section 8's benchmark before trusting this size. ----
M_GRID = [3, 5, 8, 11]                       # embedding dimension
TARGET_DENSITY_GRID = [0.02, 0.05, 0.10, 0.15]  # recurrence matrix density (find_epsilon's target)
W_SIZE_GRID  = [20, 50, 100]                 # Laplacian-eigenmap window size (points)
W_INCRE_GRID = [2, 5, 10]                    # Laplacian-eigenmap window increment (points)

# ---- Grid: cheap stage (smoothing + threshold detection) -- reuses the cached FI
# series above, so these are free to grid more finely. ----
BLOCK_SIZE_GRID = [1, 3, 5, 10]              # smooth_series block size
ALPHA_GRID = [0.10, 0.05, 0.02, 0.01]        # -> lower=100*alpha/2, upper=100*(1-alpha/2)

PARAM_COLS = ['m', 'target_density', 'w_size', 'w_incre', 'block_size', 'alpha']
GRIDS = dict(m=M_GRID, target_density=TARGET_DENSITY_GRID, w_size=W_SIZE_GRID,
             w_incre=W_INCRE_GRID, block_size=BLOCK_SIZE_GRID, alpha=ALPHA_GRID)

## Section 2 -- Data and ground truth

In [ ]:
def transition_times_from_states(time, states):
    """True transition times (both directions) from a state-label array.

    Mirrors the same helper in the KS/Pelt/Dynp notebooks: any stadial/interstadial
    change counts as ground truth, regardless of direction.
    """
    change_idx = np.where(np.diff(states) != 0)[0] + 1
    return time[change_idx]


d = np.load(DATA_PATH)
time_arr   = d['time']
db_noisy   = d['db_noisy']
state_true = d['state_true']
n_seeds    = db_noisy.shape[0]

gt_times = [transition_times_from_states(time_arr, state_true[s]) for s in range(n_seeds)]
n_transitions = [len(g) for g in gt_times]
dt = float(np.median(np.diff(time_arr)))
print(f"Loaded {DATA_PATH}: {n_seeds} seeds, {len(time_arr)} points/series, dt~={dt:.4f}")
print(f"Ground-truth transitions per seed: min={min(n_transitions)}, max={max(n_transitions)}, "
      f"mean={np.mean(n_transitions):.1f}")

## Section 3 -- Detection wrapper and per-combo scoring

`score_one` (and the expensive, disk-cached `compute_fi_series` it calls) live in
`lerm_scoring.py`, not inline -- both for the same cache-path-stability reason as
`dynp_scoring.py`, and because that module is where the staged-caching split (Section 0)
actually happens.

In [ ]:
from lerm_scoring import score_one


def _score_one(params, seed):
    return score_one(DATA_PATH, params, params['alpha'], seed, DETECTION_TOLERANCE,
                      EPS_START_BASE, M_REFERENCE, DENSITY_TOLERANCE, SEARCH_AMP,
                      N_SAMPLES_BOOTSTRAP, W_FRACTION, W_MIN)


# Warm up the cache once from the main process before any Parallel(...) dispatch --
# avoids a joblib+loky race where many worker processes simultaneously try to create
# the cache's metadata folder on its very first use (see dynp_scoring.py precedent).
# This is a real grid point too, so it's not wasted work -- after this first-ever run
# it comes straight back from cache.
_warmup_params = dict(m=M_GRID[0], target_density=TARGET_DENSITY_GRID[0],
                       w_size=W_SIZE_GRID[0], w_incre=W_INCRE_GRID[0],
                       block_size=BLOCK_SIZE_GRID[0], alpha=ALPHA_GRID[0])
_t0 = time.perf_counter()
print(_score_one(_warmup_params, 0))
print(f"Warm-up call: {time.perf_counter() - _t0:.1f}s")

## Section 4 -- Train/test split

Same `RANDOM_SEED` and split logic as KS/Pelt/Dynp, so seeds line up for cross-method
comparison in `compare_methods.ipynb`.

In [ ]:
def seed_pool(n_seeds_total, pool_size, random_state=RANDOM_SEED):
    """Fixed subsample of `pool_size` seed indices out of n_seeds_total."""
    rng = np.random.RandomState(random_state)
    return np.sort(rng.choice(n_seeds_total, size=pool_size, replace=False))


def outer_split(pool, n_train, n_test, random_state=RANDOM_SEED):
    """Split `pool` into disjoint train/test seed-index arrays, once."""
    rng = np.random.RandomState(random_state)
    shuffled = rng.permutation(pool)
    train_idx = np.sort(shuffled[:n_train])
    test_idx = np.sort(shuffled[n_train:n_train + n_test])
    return train_idx, test_idx


pool = seed_pool(n_seeds, BUDGET['pool_size'])
train_idx, test_idx = outer_split(pool, BUDGET['n_train'], BUDGET['n_test'])
print(f"train seeds ({len(train_idx)}): {train_idx}")
print(f"test seeds  ({len(test_idx)}): {test_idx}")

## Section 5 -- Grid search (train pool) + held-out evaluation (test pool) + alpha sweep

Same shared machinery as the KS notebook's Section 5, generalized to `PARAM_COLS` (6 columns:
5 structural + `alpha`).

In [ ]:
def grid_search_train(param_grid, train_idx):
    """Score every (param, seed in train_idx) combo, aggregate per param, pick the
    winner by mean f1 over the train pool only."""
    jobs = [(pi, params, s) for pi, params in enumerate(param_grid) for s in train_idx]
    scored = Parallel(n_jobs=N_JOBS)(
        delayed(_score_one)(params, s) for pi, params, s in jobs
    )
    rows = []
    for (pi, params, s), res in zip(jobs, scored):
        row = dict(params)
        row.update(pi=pi, seed=int(s), split='train', **res)
        rows.append(row)
    train_raw = pd.DataFrame(rows)

    train_summary = (train_raw.groupby(PARAM_COLS)
                      .agg(mean_precision=('precision', 'mean'), std_precision=('precision', 'std'),
                           mean_recall=('recall', 'mean'), std_recall=('recall', 'std'),
                           mean_f1=('f1', 'mean'), std_f1=('f1', 'std'),
                           mean_fp=('false_positives', 'mean'))
                      .reset_index())

    best_row = train_summary.loc[train_summary['mean_f1'].idxmax()]
    best_params = {c: best_row[c] for c in PARAM_COLS}
    return best_params, best_row, train_summary, train_raw


def evaluate_test(best_params, test_idx):
    """Apply best_params to the untouched-until-now test pool, once."""
    scored = Parallel(n_jobs=N_JOBS)(
        delayed(_score_one)(best_params, s) for s in test_idx
    )
    rows = []
    for s, res in zip(test_idx, scored):
        row = dict(best_params)
        row.update(seed=int(s), split='test', **res)
        rows.append(row)
    test_raw = pd.DataFrame(rows)
    return (test_raw['precision'].mean(), test_raw['recall'].mean(),
            test_raw['f1'].mean(), test_raw['false_positives'].mean(), test_raw)


def alpha_sweep_test(fixed_params, alpha_grid, test_idx):
    """Sweep alpha at the winning m/target_density/w_size/w_incre/block_size, test
    seeds only -- the precision/recall tradeoff curve for LERM (alpha is its PR-curve
    axis, same role d_c plays for KS). Cheap: reuses the cached FI series for every
    alpha value."""
    other = {k: v for k, v in fixed_params.items() if k != 'alpha'}
    jobs = [(a, s) for a in alpha_grid for s in test_idx]
    scored = Parallel(n_jobs=N_JOBS)(
        delayed(_score_one)(dict(other, alpha=a), s) for a, s in jobs
    )
    rows = []
    for (a, s), res in zip(jobs, scored):
        rows.append(dict(other, alpha=a, seed=int(s), split='test_sweep', **res))
    sweep_raw = pd.DataFrame(rows)
    sweep_summary = (sweep_raw.groupby('alpha')
                      .agg(mean_precision=('precision', 'mean'), mean_recall=('recall', 'mean'),
                           mean_f1=('f1', 'mean'), mean_fp=('false_positives', 'mean'))
                      .reset_index())
    return sweep_summary, sweep_raw

## Section 6 -- Per-parameter sensitivity ("which parameters are actually effective")

Same main-effects approach as the KS notebook's Section 6: for each parameter, how much does
`mean_f1` move across its tested range once every other parameter is averaged out.

In [ ]:
def param_main_effects(train_summary, param_cols, metric='mean_f1'):
    """Per-parameter marginal mean of `metric`, averaged over every other parameter.
    Returns {param_name: DataFrame(value, mean_metric)}."""
    effects = {}
    for p in param_cols:
        eff = (train_summary.groupby(p)[metric].mean()
               .reset_index().rename(columns={metric: f'mean_{metric}'}))
        effects[p] = eff
    return effects


def effect_size_ranking(effects, metric='mean_f1'):
    """Rank parameters by how much their marginal mean metric moves across the
    tested range (max - min of the main-effects curve)."""
    col = f'mean_{metric}'
    rows = [dict(param=p, effect_range=eff[col].max() - eff[col].min())
            for p, eff in effects.items()]
    return pd.DataFrame(rows).sort_values('effect_range', ascending=False).reset_index(drop=True)

## Section 7 -- Plot helpers

In [ ]:
def plot_alpha_sweep(sweep_summary, conclusion_row):
    """Test-set precision/recall tradeoff (alpha swept at the winning structural params)."""
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot(sweep_summary['mean_recall'], sweep_summary['mean_precision'], color='#2a78d6',
             linewidth=1.5, marker='o', markersize=4, zorder=2)
    ax.scatter([conclusion_row['test_recall']], [conclusion_row['test_precision']], color='#e34948',
               s=90, marker='*', zorder=5, label='selected alpha')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel('recall')
    ax.set_ylabel('precision')
    ax.set_title("LERM: TEST precision/recall tradeoff (alpha swept)")
    ax.legend(frameon=False)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(color='#e1e0d9', linewidth=0.6)
    ax.set_axisbelow(True)
    plt.tight_layout()
    fig.savefig("lerm_grid_red_snr3_alpha_sweep.png", dpi=200, bbox_inches='tight')
    plt.show()


def plot_main_effects(effects, best_params, metric='mean_f1'):
    """One small subplot per parameter: its tested values on x, marginal mean_f1 on
    y (averaged over every other parameter)."""
    col = f'mean_{metric}'
    params = list(effects.keys())
    ncols = 3
    nrows = int(np.ceil(len(params) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.4 * nrows), squeeze=False)

    for idx, p in enumerate(params):
        ax = axes[idx // ncols][idx % ncols]
        eff = effects[p]
        ax.plot(eff[p], eff[col], color='#2a78d6', linewidth=1.5, marker='o', markersize=4)
        ax.axvline(best_params[p], color='#e34948', linestyle='--', linewidth=1, alpha=0.7)
        ax.set_title(p, fontsize=10)
        ax.set_xlabel(p, fontsize=8)
        ax.set_ylabel(f'marginal {metric}', fontsize=8)
        ax.grid(color='#e1e0d9', linewidth=0.6)
        ax.set_axisbelow(True)

    for j in range(len(params), nrows * ncols):
        axes[j // ncols][j % ncols].axis('off')

    fig.suptitle("Per-parameter main effects (TRAIN grid, marginalized over the other 5 params)", y=1.0)
    plt.tight_layout()
    fig.savefig("lerm_grid_red_snr3_main_effects.png", dpi=200, bbox_inches='tight')
    plt.show()

## Section 8 -- Benchmark, then run the grid

Two-tier estimate, unlike KS's single-call timing: one timed call to the expensive, cached
`compute_fi_series` stage, and one timed call to the cheap `score_one` wrapper *on top of an
already-warm cache* (from Section 3's warm-up). The full grid's cost is dominated by however many
*unique* `(m, target_density, w_size, w_incre, seed)` combinations exist -- not by the total
number of scored rows, most of which are cheap `block_size`/`alpha` reuse of an already-cached FI
series. **Check both estimates against your actual CARC allocation before launching** -- if the
expensive-stage estimate is too large, shrink `M_GRID`/`TARGET_DENSITY_GRID`/`W_SIZE_GRID`/
`W_INCRE_GRID` (Section 1); `BLOCK_SIZE_GRID`/`ALPHA_GRID` are free to leave alone since they don't
add expensive-stage calls.

In [ ]:
param_grid = [dict(m=m, target_density=td, w_size=ws, w_incre=wi, block_size=bs, alpha=a)
              for m in M_GRID for td in TARGET_DENSITY_GRID
              for ws in W_SIZE_GRID for wi in W_INCRE_GRID
              for bs in BLOCK_SIZE_GRID for a in ALPHA_GRID]

n_expensive_combos = len(M_GRID) * len(TARGET_DENSITY_GRID) * len(W_SIZE_GRID) * len(W_INCRE_GRID)
n_expensive_calls = n_expensive_combos * len(train_idx)
n_scored_calls = len(param_grid) * len(train_idx)
print(f"Grid: {len(param_grid)} param combos x {len(train_idx)} train seeds = {n_scored_calls} scored calls")
print(f"Of which {n_expensive_combos} unique (m, target_density, w_size, w_incre) combos x "
      f"{len(train_idx)} train seeds = {n_expensive_calls} trigger a fresh compute_fi_series call")

# Time one *new* expensive-stage call (different params than the Section 3 warm-up, so
# this isn't a cache hit) and one cheap call on top of an already-warm cache.
from lerm_scoring import compute_fi_series

_probe = param_grid[len(param_grid) // 2]
_t0 = time.perf_counter()
compute_fi_series(DATA_PATH, int(train_idx[1]), _probe['m'], _probe['target_density'],
                   _probe['w_size'], _probe['w_incre'], EPS_START_BASE, M_REFERENCE,
                   DENSITY_TOLERANCE, SEARCH_AMP)
_expensive_call_s = time.perf_counter() - _t0

_t0 = time.perf_counter()
_score_one(_probe, int(train_idx[1]))   # same (m, target_density, w_size, w_incre, seed) -> cache hit
_cheap_call_s = time.perf_counter() - _t0

_cpu_count = os.cpu_count() if N_JOBS == -1 else N_JOBS
_est_expensive_h = n_expensive_calls * _expensive_call_s / 3600
_est_cheap_h = n_scored_calls * _cheap_call_s / 3600
_est_total_serial_h = _est_expensive_h + _est_cheap_h
_est_total_parallel_h = _est_total_serial_h / max(_cpu_count, 1)
print(f"Timed expensive-stage call: {_expensive_call_s:.2f}s (params={_probe})")
print(f"Timed cheap call (cache hit): {_cheap_call_s:.3f}s")
print(f"Estimated: {_est_total_serial_h:.1f}h serial, ~{_est_total_parallel_h:.2f}h on {_cpu_count} cores "
      f"(assumes near-linear joblib scaling -- sanity check before trusting this)")

In [ ]:
t0 = time.perf_counter()
best_params, train_row, train_summary, train_raw = grid_search_train(param_grid, train_idx)
test_p, test_r, test_f1, test_fp, test_raw = evaluate_test(best_params, test_idx)
sweep_summary, sweep_raw = alpha_sweep_test(best_params, ALPHA_GRID, test_idx)

edges = [p for p in PARAM_COLS if best_params[p] in (min(GRIDS[p]), max(GRIDS[p]))]

conclusion_row = dict(
    method='LERM',
    **{f'best_{p}': best_params[p] for p in PARAM_COLS},
    train_f1=train_row['mean_f1'], train_precision=train_row['mean_precision'],
    train_recall=train_row['mean_recall'],
    test_f1=test_f1, test_precision=test_p, test_recall=test_r, test_fp=test_fp,
    edge_params=",".join(edges) if edges else "none",
    n_train_seeds=len(train_idx), n_test_seeds=len(test_idx),
    train_seeds=",".join(map(str, train_idx)), test_seeds=",".join(map(str, test_idx)),
)

effects = param_main_effects(train_summary, PARAM_COLS)
ranking = effect_size_ranking(effects)

elapsed = time.perf_counter() - t0
edge_note = f"  <-- ON GRID EDGE: {edges}" if edges else ""
print(f"LERM: {best_params}  train_f1={train_row['mean_f1']:.3f}  test_f1={test_f1:.3f}  "
      f"({elapsed/3600:.2f}h){edge_note}")
print("\nEffect-size ranking (max-min of marginal mean_f1 across each param's tested range):")
print(ranking.to_string(index=False))

In [ ]:
plot_alpha_sweep(sweep_summary, conclusion_row)
plot_main_effects(effects, best_params)

## Section 9 -- Save raw / summary / conclusion CSVs

In [ ]:
# raw: every scored (param, seed) row, tagged by split -- 'train' (full grid on the
# train pool, what selection used), 'test' (the single winning combo on the test
# pool), 'test_sweep' (alpha swept at the winning other params, for the PR-curve plot)
raw_df = pd.concat([train_raw, test_raw, sweep_raw], ignore_index=True, sort=False)
RAW_CSV = "lerm_grid_red_snr3_raw.csv"
raw_df.to_csv(RAW_CSV, index=False)
print(f"Saved {len(raw_df)} rows to {RAW_CSV}")

SUMMARY_CSV = "lerm_grid_red_snr3_summary.csv"
train_summary.to_csv(SUMMARY_CSV, index=False)
print(f"Saved {len(train_summary)} rows to {SUMMARY_CSV}")

CONCLUSION_CSV = "lerm_grid_red_snr3_conclusion.csv"
pd.DataFrame([conclusion_row]).to_csv(CONCLUSION_CSV, index=False)

RANKING_CSV = "lerm_grid_red_snr3_param_ranking.csv"
ranking.to_csv(RANKING_CSV, index=False)
print(f"Saved parameter effect-size ranking to {RANKING_CSV}")

In [ ]:
print("=" * 92)
print("CONCLUSION: LERM on red/snr3 -- train/test split, joint grid over 6 parameters")
print("=" * 92)
gap = conclusion_row['train_f1'] - conclusion_row['test_f1']
print(f"best params: { {p: best_params[p] for p in PARAM_COLS} }")
print(f"train_f1={conclusion_row['train_f1']:.3f}  test_f1={conclusion_row['test_f1']:.3f}  (gap={gap:+.3f})")
print(f"test_precision={conclusion_row['test_precision']:.3f}  test_recall={conclusion_row['test_recall']:.3f}")

if edges:
    print(f"\n{len(edges)} parameter(s) landed on a grid edge: {edges} -- widen those ranges and "
          f"rerun before trusting them as converged optima.")
else:
    print(f"\nAll {len(PARAM_COLS)} parameters converged (train optimum interior to their tested range).")

print(f"\nMost effective parameter (largest main-effect swing in mean_f1): "
      f"'{ranking.iloc[0]['param']}' (range={ranking.iloc[0]['effect_range']:.3f})")
print(f"Least effective: '{ranking.iloc[-1]['param']}' (range={ranking.iloc[-1]['effect_range']:.3f})")